## Visualizing Mouse and Cricket Hunting Traces — Overlay Video

This notebook generates an overlay video of hunt trajectories on top of the source MP4, plus a fade to a static summary plot.

**Workflow**
1. Opens a folder picker (Tkinter GUI).
2. Finds one CSV (`*_analysis_full.csv` preferred, fallback `*_pythonAnalysis.csv`).
3. Finds candidate source videos (`.mp4`), excluding `*_overlay.mp4` and macOS `._*` sidecar files.
4. Loads tracking data, identifies capture boundary (`captured`), and applies a frame window `[start_frame, capture_frame)`.
5. Builds per-frame trajectory overlays (mouse + cricket) and writes `*_overlay.mp4`.
6. Adds a fade transition from the last frame to a static trace plot.
7. Saves a provenance report `*_overlay_source.txt` with source video, CSV, and transform settings.

**Adjustable Parameters (Step 2 cell)**
- `start_frame`: first frame to include in video and trails.
- `warp_to_arena`: enable/disable perspective warping to arena coordinates.
- `manual_rotation_deg`: rotate output frames (`0/90/180/270`).
- `rotate_points_with_frame`: rotate overlay coordinates with frame rotation.
- `trail_thickness`, `border_thickness`, `fade_duration_seconds`, `padding_pixels`, colors.

**Outputs**
- `*_overlay.mp4` in the selected folder.
- `*_overlay_source.txt` describing exactly which source files/settings were used.

**Notes**
- Use the source report file to debug orientation/path alignment issues.
- Arena-warp mode expects valid corner coordinates (`corners_x`, `corners_y`) in the CSV.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tkinter as tk
from tkinter import filedialog
import os
import glob
import cv2

### Step 1: Select the Data Folder
This cell is updated to find the video file in addition to the CSV file. It will automatically ignore any video files that already have the `_overlay` suffix.

In [9]:
# Set up the root Tkinter window
root = tk.Tk()
root.withdraw() # Hide the main window

# --- Set the initial directory ---
if 'last_folder_path' in locals() and os.path.isdir(last_folder_path):
    initial_dir = last_folder_path
else:
    initial_dir = os.getcwd()

# Open the dialog to ask for a directory
print("Please select the folder containing your analysis files...")
folder_path = filedialog.askdirectory(
    title='Select Folder',
    initialdir=initial_dir
)

video_path = None
if not folder_path:
    print("No folder selected. Notebook execution stopped.")
else:
    print(f"Selected folder: {folder_path}")
    # --- Save the newly selected folder path for the next run ---
    last_folder_path = folder_path
    %store last_folder_path
    
    # --- Find the original MP4 video file ---
    # Find all mp4 files, then exclude any with '_overlay'
    all_mp4_files = sorted(glob.glob(os.path.join(folder_path, '*.mp4')))
    original_mp4_files = [
        f for f in all_mp4_files
        if '_overlay' not in os.path.basename(f) and not os.path.basename(f).startswith('._')
    ]
    
    print("Candidate source videos (after filtering):")
    for i, f in enumerate(original_mp4_files):
        print(f"  [{i}] {os.path.abspath(f)}")

    if not original_mp4_files:
        print("Warning: No original .mp4 file found in the selected folder.")
    else:
        video_path = original_mp4_files[0]
        print(f"Video file found: {os.path.basename(video_path)}")
        print(f"Using source video: {os.path.abspath(video_path)}")

Please select the folder containing your analysis files...
Selected folder: /Users/kerschensteinerd/Library/CloudStorage/Dropbox/Figures/2023/ipsi_alphas/behavior/tracked_data/SERTtdt/B1677#6M/T5
Stored 'last_folder_path' (str)
Video file found: B1677#6M_D5_T5.mp4
Selected folder: /Users/kerschensteinerd/Library/CloudStorage/Dropbox/Figures/2023/ipsi_alphas/behavior/tracked_data/SERTtdt/B1677#6M/T5
Stored 'last_folder_path' (str)
Video file found: B1677#6M_D5_T5.mp4


### Step 2: Load Data, Generate Static Plot, and Create Annotated Video
This cell finds the CSV file, processes the tracking data, generates a static trace plot (used for the fade-in), then writes the annotated video frame-by-frame using a perspective transformation to warp the raw footage into a standardized arena view.

In [10]:
if folder_path and video_path: # Proceed only if a folder and video were selected
    try:
        # --- Adjustable Parameters ---
        arena_width_cm = 45
        arena_height_cm = 38
        trail_thickness = 5 # UPDATED: Increased default thickness
        border_thickness = 5 # NEW: Added adjustable border thickness
        fade_duration_seconds = 1.0
        padding_pixels = 40
        start_frame = 55  # Starting frame for video and traces (0 = beginning)
        
        warp_to_arena = False  # False keeps original orientation/geometry
        manual_rotation_deg = 180  # Set to 0 if your source already appears upright
        rotate_points_with_frame = True  # Keep True when manual_rotation_deg is non-zero

        mouse_color_rgb = [0, 0, 0]   # Dull Yellow
        cricket_color_rgb = [100, 100, 100] # Gray
        
        # --- File Handling & Data Prep ---
        primary_pattern = os.path.join(folder_path, '*_analysis_full.csv')
        csv_files = glob.glob(primary_pattern)
        if not csv_files:
            fallback_pattern = os.path.join(folder_path, '*_pythonAnalysis.csv')
            csv_files = glob.glob(fallback_pattern)
        if not csv_files:
            raise FileNotFoundError("Could not find a file ending with '_analysis_full.csv' or '_pythonAnalysis.csv'.")
        
        csv_path = csv_files[0]
        print(f"Loading data from: {os.path.basename(csv_path)}")
        
        df = pd.read_csv(csv_path)
        capture_frame = df[df['captured'] == 1].index.min()
        if pd.isna(capture_frame):
            capture_frame = len(df)
        
        # Apply start_frame constraint
        effective_start = max(start_frame, 0)
        effective_end = max(capture_frame, effective_start + 1)  # Ensure at least 1 frame
        
        print(f"Processing frames {effective_start} to {effective_end} (capture at frame {capture_frame})")
        plot_df = df.iloc[effective_start:effective_end].copy()
        
        plot_df['madj_x'] = plot_df['madj_x'].clip(0, arena_width_cm)
        plot_df['madj_y'] = plot_df['madj_y'].clip(0, arena_height_cm)
        plot_df['cadj_x'] = plot_df['cadj_x'].clip(0, arena_width_cm)
        plot_df['cadj_y'] = plot_df['cadj_y'].clip(0, arena_height_cm)

        # --- Video and Transformation Setup ---
        video_in = cv2.VideoCapture(video_path)
        fps = video_in.get(cv2.CAP_PROP_FPS)
        source_width = int(video_in.get(cv2.CAP_PROP_FRAME_WIDTH))
        source_height = int(video_in.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if source_width <= 0 or source_height <= 0:
            raise RuntimeError('Could not read source video dimensions.')

        def _normalize_rotation(deg):
            deg = int(round(deg)) % 360
            if deg in (0, 90, 180, 270):
                return deg
            return 0

        def _rotate_points(pts, width, height, rotation_deg):
            out = pts.copy()
            valid = (out[:, 0] >= 0) & (out[:, 1] >= 0)
            if not np.any(valid):
                return out
            x = out[valid, 0].astype(np.float32)
            y = out[valid, 1].astype(np.float32)
            if rotation_deg == 90:
                out[valid, 0] = np.round(height - 1 - y).astype(np.int32)
                out[valid, 1] = np.round(x).astype(np.int32)
            elif rotation_deg == 180:
                out[valid, 0] = np.round(width - 1 - x).astype(np.int32)
                out[valid, 1] = np.round(height - 1 - y).astype(np.int32)
            elif rotation_deg == 270:
                out[valid, 0] = np.round(y).astype(np.int32)
                out[valid, 1] = np.round(width - 1 - x).astype(np.int32)
            return out

        applied_rotation_deg = _normalize_rotation(manual_rotation_deg)
        if not warp_to_arena and applied_rotation_deg in (90, 270):
            frame_width, frame_height = source_height, source_width
        else:
            frame_width, frame_height = source_width, source_height
        print(f"Applying rotation to frames before writing: {applied_rotation_deg} deg")
        
        arena_aspect_ratio = arena_width_cm / arena_height_cm
        if warp_to_arena:
            drawable_width = frame_width - 2 * padding_pixels
            drawable_height = frame_height - 2 * padding_pixels
            drawable_aspect_ratio = drawable_width / drawable_height

            if drawable_aspect_ratio > arena_aspect_ratio:
                target_h = drawable_height
                target_w = target_h * arena_aspect_ratio
                padding_x = (frame_width - target_w) / 2
                padding_y = padding_pixels
            else:
                target_w = drawable_width
                target_h = target_w / arena_aspect_ratio
                padding_y = (frame_height - target_h) / 2
                padding_x = padding_pixels
        else:
            target_w = frame_width
            target_h = frame_height
            padding_x = 0
            padding_y = 0

        # --- 1. Generate and Save the Aligned Static Plot ---
        print("Generating aligned static plot for fade-in...")
        mouse_color_plt = np.array(mouse_color_rgb) / 255.0
        cricket_color_plt = np.array(cricket_color_rgb) / 255.0
        
        dpi = 100
        trail_thickness_pt = trail_thickness * (72 / dpi)
        # UPDATED: Use direct pixel-to-point conversion to match OpenCV thickness exactly
        # At 100 DPI: 1 pixel = 0.72 points, so multiply by 0.72 instead of (72/dpi)
        border_thickness_pt = border_thickness * 0.72

        fig = plt.figure(figsize=(frame_width / dpi, frame_height / dpi), dpi=dpi)
        fig.patch.set_facecolor('white')

        ax_pos = [padding_x / frame_width, padding_y / frame_height, target_w / frame_width, target_h / frame_height]
        ax = fig.add_axes(ax_pos)
        
        if warp_to_arena:
            ax.add_patch(patches.Rectangle((0, 0), arena_width_cm, arena_height_cm, facecolor='white', zorder=0))
            ax.plot(plot_df['madj_x'], plot_df['madj_y'], color=mouse_color_plt, linewidth=trail_thickness_pt, zorder=2)
            ax.plot(plot_df['cadj_x'], plot_df['cadj_y'], color=cricket_color_plt, linewidth=trail_thickness_pt, zorder=2)
            arena_border = patches.Rectangle((0, 0), arena_width_cm, arena_height_cm, linewidth=border_thickness_pt, edgecolor='black', facecolor='none', zorder=3)
            ax.add_patch(arena_border)
            ax.set_xlim(0, arena_width_cm)
            ax.set_ylim(0, arena_height_cm)
        else:
            ax.add_patch(patches.Rectangle((0, 0), frame_width, frame_height, facecolor='white', zorder=0))
            ax.plot(plot_df['mid_x'], plot_df['mid_y'], color=mouse_color_plt, linewidth=trail_thickness_pt, zorder=2)
            ax.plot(plot_df['cricket_x'], plot_df['cricket_y'], color=cricket_color_plt, linewidth=trail_thickness_pt, zorder=2)
            ax.set_xlim(0, frame_width)
            ax.set_ylim(0, frame_height)
        ax.invert_yaxis()
        ax.axis('off')
        
        temp_plot_path = os.path.join(folder_path, 'temp_plot_for_video.png')
        plt.savefig(temp_plot_path, format='png', dpi=dpi, facecolor='white')
        plt.close(fig)

        # --- 2. Prepare for Video Generation ---
        print("Starting video generation...")
        print(f"Overlay source video: {os.path.abspath(video_path)}")
        base_video_name = os.path.splitext(os.path.basename(video_path))[0]
        video_out_path = os.path.join(folder_path, f"{base_video_name}_overlay.mp4")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_out = cv2.VideoWriter(video_out_path, fourcc, fps, (frame_width, frame_height))
        
        mouse_color_bgr = mouse_color_rgb[::-1]
        cricket_color_bgr = cricket_color_rgb[::-1]

        transform_matrix = None
        if warp_to_arena:
            corners = np.float32(df.loc[:3, ['corners_x', 'corners_y']].values)
            video_corners = np.float32([corners[0], corners[1], corners[2], corners[3]])
            target_corners = np.float32([[padding_x, padding_y], [padding_x + target_w, padding_y], [padding_x, padding_y + target_h], [padding_x + target_w, padding_y + target_h]])
            transform_matrix = cv2.getPerspectiveTransform(video_corners, target_corners)
        
        # --- 4. Generate Video Frames with Overlay ---
        if warp_to_arena:
            scale_x = target_w / arena_width_cm
            scale_y = target_h / arena_height_cm
            mouse_pts_scaled = plot_df[['madj_x', 'madj_y']].values * [scale_x, scale_y]
            mouse_pts = (mouse_pts_scaled + [padding_x, padding_y]).astype(np.int32)
            cricket_pts_scaled = plot_df[['cadj_x', 'cadj_y']].values * [scale_x, scale_y]
            cricket_pts = (cricket_pts_scaled + [padding_x, padding_y]).astype(np.int32)
        else:
            mouse_pts = plot_df[['mid_x', 'mid_y']].fillna(-1).values.astype(np.int32)
            cricket_pts = plot_df[['cricket_x', 'cricket_y']].fillna(-1).values.astype(np.int32)
            if rotate_points_with_frame:
                mouse_pts = _rotate_points(mouse_pts, source_width, source_height, applied_rotation_deg)
                cricket_pts = _rotate_points(cricket_pts, source_width, source_height, applied_rotation_deg)
        
        # Skip to start frame in video
        video_in.set(cv2.CAP_PROP_POS_FRAMES, effective_start)
        
        last_frame = None
        frames_to_process = effective_end - effective_start
        for i in range(frames_to_process):
            ret, frame = video_in.read()
            if not ret: break
            
            if warp_to_arena and transform_matrix is not None:
                output_frame = cv2.warpPerspective(frame, transform_matrix, (frame_width, frame_height))
            else:
                if applied_rotation_deg == 90:
                    output_frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
                elif applied_rotation_deg == 180:
                    output_frame = cv2.rotate(frame, cv2.ROTATE_180)
                elif applied_rotation_deg == 270:
                    output_frame = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
                else:
                    output_frame = frame.copy()
            
            # Draw traces first, then border on top (same order as matplotlib plot)
            cv2.polylines(output_frame, [mouse_pts[:i+1]], isClosed=False, color=mouse_color_bgr, thickness=trail_thickness)
            cv2.polylines(output_frame, [cricket_pts[:i+1]], isClosed=False, color=cricket_color_bgr, thickness=trail_thickness)
            # Border drawn last so it appears on top (arena-warp mode only)
            if warp_to_arena:
                cv2.rectangle(output_frame, (int(padding_x), int(padding_y)), (int(padding_x + target_w), int(padding_y + target_h)), (0, 0, 0), border_thickness)
            
            video_out.write(output_frame)
            last_frame = output_frame
            if i % 100 == 0: print(f"  ...processed frame {i}/{frames_to_process}")

        # --- 5. Add Fade-in Effect ---
        print("Adding fade-in effect...")
        if last_frame is not None:
            plot_img = cv2.imread(temp_plot_path)
            if not warp_to_arena:
                if applied_rotation_deg == 90:
                    plot_img = cv2.rotate(plot_img, cv2.ROTATE_90_CLOCKWISE)
                elif applied_rotation_deg == 180:
                    plot_img = cv2.rotate(plot_img, cv2.ROTATE_180)
                elif applied_rotation_deg == 270:
                    plot_img = cv2.rotate(plot_img, cv2.ROTATE_90_COUNTERCLOCKWISE)
            
            fade_frames = int(fps * fade_duration_seconds)
            for i in range(fade_frames):
                alpha = i / fade_frames
                beta = 1.0 - alpha
                faded_frame = cv2.addWeighted(last_frame, beta, plot_img, alpha, 0.0)
                video_out.write(faded_frame)
        
        for _ in range(int(fps)):
             video_out.write(plot_img)

        # --- 6. Clean Up ---
        video_in.release()
        video_out.release()
        os.remove(temp_plot_path)
        source_report_path = os.path.join(folder_path, f"{base_video_name}_overlay_source.txt")
        with open(source_report_path, 'w') as f:
            f.write(f"source_video={os.path.abspath(video_path)}\n")
            f.write(f"overlay_video={os.path.abspath(video_out_path)}\n")
            f.write(f"csv_used={os.path.abspath(csv_path)}\n")
            f.write(f"warp_to_arena={warp_to_arena}\n")
            f.write(f"manual_rotation_deg={manual_rotation_deg}\n")
            f.write(f"applied_rotation_deg={applied_rotation_deg}\n")
            f.write(f"rotate_points_with_frame={rotate_points_with_frame}\n")
            f.write(f"frame_size_out={frame_width}x{frame_height}\n")
            f.write(f"effective_start={effective_start}\n")
            f.write(f"effective_end={effective_end}\n")
        print(f"\n✅ Video saved successfully to: {video_out_path}")
        print(f"Overlay source report saved to: {source_report_path}")

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'video_in' in locals() and video_in.isOpened(): video_in.release()
        if 'video_out' in locals() and video_out.isOpened(): video_out.release()
        if 'temp_plot_path' in locals() and os.path.exists(temp_plot_path): os.remove(temp_plot_path)

Loading data from: B1677#6M_D5_T5DeepCut_resnet50_CricketMar21shuffle1_100000_analysis_full.csv
Processing frames 55 to 259 (capture at frame 259)
Generating aligned static plot for fade-in...
Starting video generation...
  ...processed frame 0/204
  ...processed frame 100/204
  ...processed frame 100/204
  ...processed frame 200/204
Adding fade-in effect...
  ...processed frame 200/204
Adding fade-in effect...

✅ Video saved successfully to: /Users/kerschensteinerd/Library/CloudStorage/Dropbox/Figures/2023/ipsi_alphas/behavior/tracked_data/SERTtdt/B1677#6M/T5/B1677#6M_D5_T5_overlay.mp4

✅ Video saved successfully to: /Users/kerschensteinerd/Library/CloudStorage/Dropbox/Figures/2023/ipsi_alphas/behavior/tracked_data/SERTtdt/B1677#6M/T5/B1677#6M_D5_T5_overlay.mp4


In [ ]:
(3.0789/45) *10

In [ ]:
# Close Tkinter cleanly at the end of the notebook
try:
    root.quit()  # safe if mainloop was started
except Exception:
    pass
root.destroy()